# Transformers: de la atención al estado del arte — **Notebook de extensión**

**Curso:** Deep Learning — Maestría y Doctorado en Ciencia de Datos · **Duración sugerida:** 5 horas · **Framework:** TensorFlow 2.x / Keras

> ### ⚠️ Cómo usar este archivo
> Este notebook contiene **únicamente el material nuevo** y las **correcciones** a tu notebook `transformer_model_copy.ipynb`. Cada bloque lleva una caja roja **📍 Punto de inserción** que indica **después de qué celda de tu original** debe ir. Tu contenido sobre Bahdanau, Luong, self-attention, multi-head y el bloque Transformer **no se modifica aquí**: se conserva tal cual en tu archivo y este material lo extiende.
>
> Las celdas de código de este notebook son **autocontenidas en orden**: ejecute de arriba hacia abajo (el Bloque 0 define las utilidades que usan los demás).

## Mapa de la sesión (5 h)

| Bloque | Contenido | Resultado formal | Tiempo |
|---|---|---|---|
| **0** | Preparación | — | 5 min |
| **A** | Atención clásica en TF | — | 25 min |
| **B** | Propiedades de la autoatención | **T1** equivarianza · **T2** escalado · **T3** rango | 45 min |
| **C** | Codificación posicional | **T4** RoPE | 30 min |
| **D** | Transformer completo + LM en TF | — | 40 min |
| **E** | Estabilidad | **T5** colapso de rango · **T6** Pre/Post-LN | 35 min |
| **F** | Atención eficiente e inferencia | **T7** softmax en línea / FlashAttention | 40 min |
| **G** | Escalamiento | **T8** compute-óptimo | 20 min |
| **H** | Expresividad y límites | **T9** universal · **T10** TC⁰ | 25 min |
| **I** | Interpretabilidad mecanicista | circuitos QK/OV, induction heads | 20 min |
| **J** | Ciencia de datos aplicada | series de tiempo y tabulares | 25 min |
| **K** | Mapa de frontera | MoE, ViT/CLIP, LoRA, RLHF/DPO, SSM | 15 min |

**Marca 🎓 Avanzado (doctorado):** secciones con demostraciones completas o resultados de frontera, omitibles en una primera lectura de maestría.

### ✅ Estado de verificación (léalo antes de enseñar)

| Elemento | Estado |
|---|---|
| Matemática de T1, T2, T3, T4, T5, T6, T7, T8 | **Verificada numéricamente** (NumPy) durante la preparación: equivarianza $\sim10^{-16}$; $\operatorname{Var}(q^\top k)=d_k$; construcción del T3 reproduce $A$ con error $10^{-15}$; RoPE constante por diagonal; atención por bloques $= $ exacta ($10^{-16}$); fórmula cerrada de T8 vs. numérica ($10^{-5}$ rel.) |
| Código **TensorFlow/Keras** de este notebook | **NO fue ejecutado** (el entorno de preparación no tenía TensorFlow). Sólo se comprobó la **sintaxis**. **Ejecútelo completo antes de la clase** y corrija lo que falle. |
| Resultados empíricos (curvas, precisiones, MSE) | **Dependen de su hardware y semillas**. Los textos no prometen valores; **reporte lo que obtenga**. |
| Bibliografía | Trabajos centrales **buscados en línea**; el resto citado de memoria (ver nota en Referencias). |
| Teoremas T5, T6, T9, T10 | Enunciados en **forma resumida**; consulte la fuente para constantes e hipótesis exactas. |

In [ ]:
# ==== Configuración global (ejecutar primero) ====
import math, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})
print("TensorFlow:", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la celda que contiene <b>«Bahdanau Attention (Additive)»</b> (celda 8 de tu original), o como nota al final de esa sección.
</div>

````{admonition} ⚠️ Corrección técnica: *teacher forcing* frente a estrategia de decodificación
:class: warning

En la fórmula para $y_t$ de tu texto las etiquetas aparecen **intercambiadas**. Lo correcto es:

- **Entrenamiento (*teacher forcing*):** el decoder recibe como entrada el token **real** $y_{t-1}^{\text{real}}$ y se minimiza $-\sum_t\log p_\theta(y_t^{\text{real}}\mid y_{<t}^{\text{real}},X)$. **No se muestrea nada.**
- **Inferencia:** el decoder recibe su **propia** salida previa, y el siguiente token se elige con una *estrategia de decodificación*:

$$
y_t=\begin{cases}
\arg\max_{v}\, p_\theta(v\mid y_{<t},X) & \text{(voraz / greedy)}\\[2pt]
y\sim p_\theta(\cdot\mid y_{<t},X) & \text{(muestreo)}\\[2pt]
\text{top-}k,\ \text{top-}p,\ \text{beam search} & \text{(variantes)}
\end{cases}
$$

La discrepancia entre entrenar con tokens reales y generar con tokens propios se llama **sesgo de exposición** (*exposure bias*) [Ranzato et al., 2016].
````

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la celda que contiene <b>«Formulación: Multi-head attention (MHA)…»</b> (celda 17 de tu original).
</div>

````{admonition} ⚠️ Precisión: el «Teorema (Yun et al., 2020)»
:class: warning

La frase *«Multi-head attention puede aproximar cualquier función continua de múltiples variables»* es **imprecisa**. El resultado correcto se enuncia en el **Teorema 9** (Bloque H): se trata de **Transformers completos** (atención **más** capas feed-forward), de funciones **secuencia-a-secuencia** continuas sobre un **dominio compacto**, con aproximación en norma $L^p$ ($1\le p<\infty$), y la clase aproximada es la de funciones **equivariantes a permutación**, salvo que se añada **codificación posicional**. Una capa de atención *sola* no es un aproximador universal.
````

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la celda que contiene <b>«Encoder Layer Matemático… Residual Connections… Ventajas sobre LSTM»</b> (celda 20 de tu original).
</div>

````{admonition} ⚠️ Precisión: «sin desvanecimiento del gradiente» requiere matices
:class: warning

La tabla afirma que en el Transformer el gradiente es «$O(1)$» y «sin desvanecimiento». Es una **simplificación imprecisa**:

1. La residual garantiza un **término identidad** en el jacobiano, $\partial y/\partial x=I+\partial F/\partial x$, pero a través de $L$ capas el gradiente es un **producto** $\prod_l(I+J_l)$ cuya norma puede **crecer o decrecer**.
2. La **posición de LayerNorm importa**: en la arquitectura original (**Post-LN**) los gradientes cerca de la salida son grandes en la inicialización y exigen *warm-up*; en **Pre-LN** están mejor condicionados (**T6**, Bloque E) [Xiong et al., 2020].
3. La autoatención *pura* sufre **colapso de rango doblemente exponencial** con la profundidad (**T5**) [Dong et al., 2021]; las residuales son precisamente lo que lo evita.

Un enunciado correcto: *la autoatención conecta cualquier par de tokens con un camino de longitud 1 (frente a $O(n)$ en una RNN) y las residuales preservan un camino directo para el gradiente; pero la estabilidad real depende de la normalización y la inicialización.*
````

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la sección <b>«Atención Luong (Multiplicativa)»</b> (celda 10 de tu original), antes de <i>## Self-Attention</i>.
</div>

---
## A. Atención clásica en TensorFlow
Implementamos las dos puntuaciones de alineamiento como capas de Keras y verificamos que producen distribuciones válidas.

In [ ]:
class BahdanauAttention(layers.Layer):
    # e_tj = v_a^T tanh(W_a s + U_a h_j)   (aditiva)
    def __init__(self, units, **kw):
        super().__init__(**kw)
        self.Wa = layers.Dense(units, use_bias=False); self.Ua = layers.Dense(units, use_bias=False); self.va = layers.Dense(1, use_bias=False)
    def call(self, s, H):                                   # s: (B,d_s)  H: (B,S,d_h)
        e = self.va(tf.tanh(self.Wa(s)[:, None, :] + self.Ua(H)))
        alpha = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(alpha * H, axis=1), tf.squeeze(alpha, -1)

class LuongAttention(layers.Layer):
    # e_tj = s^T W_a h_j   (multiplicativa "general")
    def __init__(self, d_h, **kw):
        super().__init__(**kw); self.Wa = layers.Dense(d_h, use_bias=False)
    def call(self, s, H):
        e = tf.einsum("bd,bsd->bs", self.Wa(s), H); alpha = tf.nn.softmax(e, axis=-1)
        return tf.einsum("bs,bsd->bd", alpha, H), alpha

B, S, dh, ds = 2, 4, 16, 16
H = tf.random.normal((B, S, dh)); s = tf.random.normal((B, ds))
for name, layer in [("Bahdanau", BahdanauAttention(32)), ("Luong", LuongAttention(dh))]:
    c, a = layer(s, H)
    print(f"{name:9s} contexto {c.shape} | pesos suman 1: {np.allclose(a.numpy().sum(-1), 1.0, atol=1e-5)} | min >= 0: {a.numpy().min() >= 0}")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la celda <b>«Relación con Bahdanau / Luong y ventajas conceptuales»</b> (celda 15 de tu original), antes de <i>## Multi-head Attention</i>.
</div>

---
## B. Propiedades matemáticas de la autoatención
**Notación.** $X\in\mathbb{R}^{n\times d}$ (una fila por token), $Q=XW_Q,\ K=XW_K,\ V=XW_V$ y
$$\operatorname{Attn}(X)=\operatorname{softmax}\!\Big(\tfrac{QK^\top}{\sqrt{d_k}}+M\Big)V,$$
con $M$ una máscara aditiva ($M=0$ sin máscara; $M_{ij}=-\infty$ si $i<j$ en el caso causal).

### Teorema 1 (Equivarianza a permutaciones)
> **Teorema 1.** Sea $\boldsymbol{P}$ una matriz de permutación y $M=0$. Entonces para cualesquiera $W_Q,W_K,W_V$,
> $$\operatorname{Attn}(\boldsymbol{P}X)=\boldsymbol{P}\,\operatorname{Attn}(X).$$
> La autoatención (y multi-head, residuales, LayerNorm y FFN por token) es **equivariante a permutaciones**. En consecuencia, **sin información posicional un Transformer sólo ve un multiconjunto de tokens**.
>
> **Corolario.** Si $x_i=x_j$, las filas $i,j$ de la salida son iguales. Con **máscara causal** el resultado **falla**: la máscara rompe la simetría y *codifica implícitamente* el orden.

**Referencias:** Vaswani et al. (2017) [1]; Lee et al. (2019) [2]; Yun et al. (2020) [3]; Haviv et al. (2022) [4].

#### Demostración
Las proyecciones actúan fila a fila, luego conmutan con $\boldsymbol{P}$: $Q'=\boldsymbol{P}Q$, $K'=\boldsymbol{P}K$, $V'=\boldsymbol{P}V$ y $Q'K'^\top=\boldsymbol{P}QK^\top\boldsymbol{P}^\top$. La softmax por filas cumple $\operatorname{softmax}(\boldsymbol{P}S\boldsymbol{P}^\top)=\boldsymbol{P}\operatorname{softmax}(S)\boldsymbol{P}^\top$ (permutar columnas sólo reordena los términos de cada normalización; permutar filas reordena las filas). Luego
$$\operatorname{Attn}(\boldsymbol{P}X)=\boldsymbol{P}\operatorname{softmax}(S)\boldsymbol{P}^\top\boldsymbol{P}V=\boldsymbol{P}\operatorname{Attn}(X),$$
usando $\boldsymbol{P}^\top\boldsymbol{P}=I$. Con máscara causal $\boldsymbol{P}M\boldsymbol{P}^\top\neq M$ en general. $\qquad\blacksquare$

In [ ]:
# ---- Verificación del Teorema 1 ----
def attn_tf(X, WQ, WK, WV, mask=None):
    Q, K, V = X @ WQ, X @ WK, X @ WV
    S = Q @ tf.transpose(K) / math.sqrt(WK.shape[-1])
    if mask is not None: S = S + mask
    return tf.nn.softmax(S, axis=-1) @ V

n, d, dk = 7, 16, 8
X  = tf.random.normal((n, d)); WQ, WK, WV = [tf.random.normal((d, dk)) / math.sqrt(d) for _ in range(3)]
P = tf.one_hot(tf.random.shuffle(tf.range(n)), n)
Y, Yp = attn_tf(X, WQ, WK, WV), attn_tf(P @ X, WQ, WK, WV)
print(f"sin máscara : ||Attn(PX) - P Attn(X)|| = {tf.norm(Yp - P @ Y).numpy():.2e}   (≈ 0)")
M = tf.constant(np.triu(np.full((n, n), -1e9, dtype="float32"), 1))
print(f"causal      : ||Attn(PX) - P Attn(X)|| = {tf.norm(attn_tf(P @ X, WQ, WK, WV, M) - P @ attn_tf(X, WQ, WK, WV, M)).numpy():.2e}   (> 0)")
Xi = tf.tile(tf.random.normal((1, d)), (n, 1)); Yi = attn_tf(Xi, WQ, WK, WV)
print("tokens idénticos -> filas idénticas:", np.allclose(Yi.numpy(), Yi.numpy()[0], atol=1e-5))

### Teorema 2 (Escalado $1/\sqrt{d_k}$: varianza y saturación de la softmax)
> **Teorema 2.** Sean $q,k\in\mathbb{R}^{d_k}$ con componentes **independientes**, media $0$ y varianza $1$. Entonces
> $$\mathbb{E}[q^\top k]=0,\qquad \operatorname{Var}(q^\top k)=d_k,\qquad \operatorname{Var}\!\Big(\tfrac{q^\top k}{\sqrt{d_k}}\Big)=1.$$
> Además, si los scores tienen desviación típica $\sigma_s\to\infty$, la softmax tiende a un vector *one-hot* y su jacobiano $J=\operatorname{diag}(p)-pp^\top$ tiende a $0$: **los gradientes se anulan**.

**Referencias:** Vaswani et al. (2017) [1, Sec. 3.2.1]; Goodfellow et al. (2016) [5]; QK-Norm: Dehghani et al. (2023) [6].

#### Demostración
**(i)** Por independencia $\mathbb{E}[q_ik_i]=0$ y $\operatorname{Var}(q_ik_i)=\mathbb{E}[q_i^2]\mathbb{E}[k_i^2]=1$; los $d_k$ sumandos son independientes, así que $\operatorname{Var}(q^\top k)=d_k$. Dividir por $\sqrt{d_k}$ divide la varianza por $d_k$.
**(ii)** Si $s_j\sim\mathcal{N}(0,\sigma_s^2)$ i.i.d., la brecha entre los dos mayores crece como $\Theta(\sigma_s)$, luego $p_{(1)}\to1$. El jacobiano cumple $\|J\|_F^2=\sum p_i^2-2\sum p_i^3+(\sum p_i^2)^2$, que vale $1-2+1=0$ si $p$ es one-hot. $\qquad\blacksquare$

*Observación.* (i) exige independencia y varianza unitaria *a la entrada*; tras entrenar hay correlaciones y el escalado deja de ser exacto, pero sigue siendo la **inicialización correcta**.

In [ ]:
# ---- Verificación del Teorema 2 ----
rng = np.random.default_rng(0)
print(f"{'d_k':>6} {'Var(q·k)':>10} {'teoría':>8} {'Var(q·k/√dk)':>14} | {'H sin esc.':>10} {'H con esc.':>10} | {'||J|| sin':>9} {'||J|| con':>9}")
for dk_ in [4, 16, 64, 256, 1024]:
    q = rng.normal(size=(20000, dk_)); k = rng.normal(size=(20000, dk_)); dots = (q * k).sum(1)
    qq = rng.normal(size=(300, dk_)); KK = rng.normal(size=(64, dk_))
    s_un = qq @ KK.T; s_sc = s_un / np.sqrt(dk_); p_un, p_sc = softmax_np(s_un), softmax_np(s_sc)
    H  = lambda p: float((-(p * np.log(p + 1e-30)).sum(-1)).mean())
    Jn = lambda p: float(np.mean([np.linalg.norm(np.diag(r) - np.outer(r, r)) for r in p]))
    print(f"{dk_:6d} {dots.var():10.1f} {dk_:8d} {(dots/np.sqrt(dk_)).var():14.3f} | {H(p_un):10.3f} {H(p_sc):10.3f} | {Jn(p_un):9.4f} {Jn(p_sc):9.4f}")
print(f"(entropía máxima con 64 claves = {np.log(64):.3f})")

### Teorema 3 (Rango de la matriz de scores) — 🎓 Avanzado
> **Teorema 3.** Sea $S=\tfrac1{\sqrt{d_k}}XW_QW_K^\top X^\top\in\mathbb{R}^{n\times n}$. Entonces $\operatorname{rank}(S)\le d_k$. Recíprocamente, si $d_k\ge n$, $d\ge n$ y $X$ tiene rango fila completo, **toda** matriz estocástica por filas con entradas positivas $A$ es representable como $A=\operatorname{softmax}(S)$ para algún $S$ de esa forma. Con $d_k<n$ sólo se representan aproximaciones de bajo rango en el espacio de *logits*.

**Referencias:** Bhojanapalli et al. (2020) [7]; Yun et al. (2020) [3].

#### Demostración
*(Cota)* $S$ es producto de matrices $n\times d,\ d\times d_k,\ d_k\times d,\ d\times n$; el rango de un producto no excede el menor rango de los factores y $\operatorname{rank}(W_QW_K^\top)\le d_k$.
*(Representabilidad)* La softmax por filas es invariante a sumar una constante por fila, así que $A=\operatorname{softmax}(S)$ sii $S=\log A+c\mathbf{1}^\top$. Tome $S^\star=\log A$ (definida porque $A>0$) y $W_QW_K^\top=\sqrt{d_k}\,X^{+}S^{\star}(X^{\top})^{+}$, con $X^+$ la pseudo-inversa; como $XX^+=I_n$ (rango fila completo) se obtiene $S=S^\star$. Esta matriz tiene rango $\le\operatorname{rank}(S^\star)\le n\le d_k$, luego puede factorizarse como $W_QW_K^\top$ con $d_k$ columnas. $\qquad\blacksquare$

In [ ]:
# ---- Verificación del Teorema 3: (a) la construcción, (b) el error al limitar d_k ----
rng = np.random.default_rng(2); n, d, dk = 8, 16, 8
X_ = rng.normal(size=(n, d)); A_ = softmax_np(rng.normal(size=(n, n)) * 2); Sstar = np.log(A_)
Xp = np.linalg.pinv(X_); W = np.sqrt(dk) * Xp @ Sstar @ Xp.T
print("(a) softmax(S) == A con la construcción:", np.allclose(softmax_np(X_ @ W @ X_.T / np.sqrt(dk)), A_), "| rank(W) =", np.linalg.matrix_rank(W), "<= n =", n)

rng = np.random.default_rng(1); n = 12
target = softmax_np(rng.normal(size=(n, n)) * 2); logT = np.log(target); logT -= logT.mean(1, keepdims=True)
U, s, Vt = np.linalg.svd(logT); errs = []
for dk_ in range(1, n + 1):
    Sk = (U[:, :dk_] * s[:dk_]) @ Vt[:dk_]; errs.append(np.abs(softmax_np(Sk) - target).max())
plt.figure(figsize=(6, 3.2)); plt.semilogy(range(1, n + 1), np.maximum(errs, 1e-17), "o-")
plt.xlabel("$d_k$"); plt.ylabel("error máximo"); plt.title("T3: el error se anula cuando $d_k \\geq \\mathrm{rank}$"); plt.show()

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque B (o antes de <i>## Transformer</i>, celda 19 de tu original).
</div>

---
## C. Codificación posicional
Por el Teorema 1 hay que inyectar el orden.

| Método | Idea | Dónde | Ref. |
|---|---|---|---|
| **Sinusoidal** | $PE_{p,2i}=\sin(p/10000^{2i/d}),\ PE_{p,2i+1}=\cos(\cdot)$ sumada al embedding | entrada | [1] |
| **Aprendida** | tabla de $n_{\max}$ vectores entrenables | entrada | [8] |
| **Relativa (Shaw)** | sesgos que dependen de $j-i$ | dentro de la atención | [9] |
| **RoPE** | rota $q,k$ por un ángulo proporcional a la posición | sobre $q,k$ | [10] |
| **ALiBi** | sesgo lineal $-m|i-j|$ sobre los scores | scores | [11] |

**Propiedad deseable:** que $q_m^\top k_n$ dependa de las posiciones sólo por $m-n$. La sinusoidal **aditiva** no la cumple exactamente (aparecen términos cruzados); RoPE sí.

### Teorema 4 (Propiedad relativa de RoPE)
> **Teorema 4.** Sea $d$ par, $\theta_i=b^{-2(i-1)/d}$ ($b=10000$) y $R_m=\operatorname{blockdiag}\big(R(m\theta_1),\dots,R(m\theta_{d/2})\big)$ con $R(\varphi)=\begin{pmatrix}\cos\varphi&-\sin\varphi\\\sin\varphi&\cos\varphi\end{pmatrix}$. Si $\tilde q_m=R_mq$ y $\tilde k_n=R_nk$, entonces
> $$\tilde q_m^\top\tilde k_n=q^\top R_{n-m}\,k,\qquad R_m^\top R_n=R_{n-m},$$
> luego el score depende de $(m,n)$ **sólo por $n-m$**. Además $R_m$ es ortogonal: $\|\tilde q_m\|=\|q\|$.

**Referencias:** Su et al. (2021) [10]; Press et al. (2022) [11]; Barbero et al. (2025) [12].

#### Demostración
Cada bloque es una rotación: $R(\alpha)^\top=R(-\alpha)$ y $R(\alpha)R(\beta)=R(\alpha+\beta)$. Por bloques, $R(m\theta_i)^\top R(n\theta_i)=R((n-m)\theta_i)$; al ser diagonal por bloques, $R_m^\top R_n=R_{n-m}$ y $\tilde q_m^\top\tilde k_n=q^\top R_m^\top R_nk=q^\top R_{n-m}k$. La ortogonalidad se sigue de la de cada $R(\varphi)$. $\qquad\blacksquare$

**Por qué funciona.** No añade parámetros; se aplica a $q,k$ y no a los *values*; conserva $\|q\|$; las frecuencias $\theta_i$ (rápidas→lentas) dan un decaimiento de la correlación con la distancia.

In [ ]:
# ---- RoPE en TF (longitud dinámica) y verificación del Teorema 4 ----
def rope_angles(n, d, base=10000.0):
    theta = tf.constant(base ** (-np.arange(0, d, 2) / d), dtype=tf.float32)
    ang = tf.einsum("n,k->nk", tf.cast(tf.range(n), tf.float32), theta)
    return tf.cos(ang), tf.sin(ang)

def apply_rope(x, cos, sin):                                         # x: (..., n, d)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    r1, r2 = x1 * cos - x2 * sin, x1 * sin + x2 * cos
    return tf.reshape(tf.stack([r1, r2], axis=-1), tf.shape(x))

d, n = 16, 64
cos, sin = rope_angles(n, d)
q, k = tf.random.normal((d,)), tf.random.normal((d,))
Q = apply_rope(tf.tile(q[None], (n, 1)), cos, sin); K = apply_rope(tf.tile(k[None], (n, 1)), cos, sin)
scores = (Q @ tf.transpose(K)).numpy()
max_dev = max(np.ptp(np.diagonal(scores, offset=o)) for o in range(-n + 1, n))
print(f"máx. desviación dentro de cada diagonal (n-m fijo): {max_dev:.2e}   (≈ 0 ⇒ sólo importa n-m)")
print(f"isometría: ||R q|| = {tf.norm(Q[37]).numpy():.6f}  vs  ||q|| = {tf.norm(q).numpy():.6f}")

# Comparación con la sinusoidal ADITIVA
def sinusoidal(n, d):
    pos = np.arange(n)[:, None]; i = np.arange(d // 2)[None]; ang = pos / (10000 ** (2 * i / d)); pe = np.zeros((n, d), "float32")
    pe[:, 0::2], pe[:, 1::2] = np.sin(ang), np.cos(ang); return pe
dm, dk_, nn = 32, 16, 40; pe = sinusoidal(nn, dm); WQ_, WK_ = np.random.randn(dm, dk_) / 6, np.random.randn(dm, dk_) / 6
xx = np.random.randn(dm); sc_add = ((xx + pe) @ WQ_) @ ((xx + pe) @ WK_).T
dev_add = max(np.ptp(np.diagonal(sc_add, offset=o)) for o in range(-nn + 1, nn) if abs(o) < nn - 2)
print(f"sinusoidal aditiva: máx. desviación por diagonal = {dev_add:.4f}  |  RoPE: {max_dev:.2e}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); im = ax[0].imshow(scores, cmap="coolwarm"); ax[0].set_title("scores RoPE: estructura Toeplitz"); plt.colorbar(im, ax=ax[0])
ax[1].plot(scores[0], label="m=0"); ax[1].plot(scores[20, 20:], "--", label="m=20 (desplazado)"); ax[1].legend(); ax[1].set_title("sólo depende de n−m"); plt.show()

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después de la celda <b>«Pseudocódigo Python / Implementación Self-Attention»</b> (celda 21 de tu original). Sustituye con este código las celdas de código vacías que siguen.
</div>

---
## D. El bloque Transformer completo en TensorFlow
Implementamos **desde cero** atención con máscara, multi-head (con **GQA**), FFN, residuales, LayerNorm y los dos órdenes de normalización:
$$\text{Post-LN: } x\mapsto \mathrm{LN}\big(x+F(x)\big)\qquad\qquad \text{Pre-LN: } x\mapsto x+F\big(\mathrm{LN}(x)\big).$$

In [ ]:
class MultiHeadAttention(layers.Layer):
    # Multi-cabeza propia con soporte GQA (n_kv_heads < n_heads), máscara causal, cross-attention y RoPE opcional.
    def __init__(self, d_model, n_heads, n_kv_heads=None, causal=False, rope=False, **kw):
        super().__init__(**kw)
        assert d_model % n_heads == 0
        self.h, self.hk = n_heads, (n_kv_heads or n_heads)
        assert self.h % self.hk == 0, "n_heads debe ser múltiplo de n_kv_heads"
        self.dh, self.causal, self.rope = d_model // n_heads, causal, rope
        self.wq = layers.Dense(self.h  * self.dh, use_bias=False)
        self.wk = layers.Dense(self.hk * self.dh, use_bias=False)
        self.wv = layers.Dense(self.hk * self.dh, use_bias=False)
        self.wo = layers.Dense(d_model, use_bias=False)

    def _split(self, x, h):                                   # (B,n,h*dh) -> (B,h,n,dh)
        B, n = tf.shape(x)[0], tf.shape(x)[1]
        return tf.transpose(tf.reshape(x, (B, n, h, self.dh)), (0, 2, 1, 3))

    def call(self, x, context=None, return_weights=False):
        ctx = x if context is None else context
        q, k, v = self._split(self.wq(x), self.h), self._split(self.wk(ctx), self.hk), self._split(self.wv(ctx), self.hk)
        if self.rope:
            cos, sin = rope_angles(tf.shape(q)[2], self.dh)   # longitud dinámica (robusto en tf.function)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        rep = self.h // self.hk
        if rep > 1: k, v = tf.repeat(k, rep, axis=1), tf.repeat(v, rep, axis=1)
        s = tf.matmul(q, k, transpose_b=True) / math.sqrt(self.dh)
        if self.causal:
            nq, nk = tf.shape(s)[-2], tf.shape(s)[-1]
            s += (1.0 - tf.linalg.band_part(tf.ones((nq, nk)), -1, 0)) * -1e9
        a = tf.nn.softmax(s, axis=-1)
        o = tf.matmul(a, v)
        o = tf.reshape(tf.transpose(o, (0, 2, 1, 3)), (tf.shape(x)[0], tf.shape(x)[1], self.h * self.dh))
        o = self.wo(o)
        return (o, a) if return_weights else o


class TransformerBlock(layers.Layer):
    def __init__(self, d_model, n_heads, d_ff, n_kv_heads=None, causal=False, rope=False,
                 pre_ln=True, dropout=0.1, cross=False, **kw):
        super().__init__(**kw)
        self.pre_ln, self.cross = pre_ln, cross
        self.attn = MultiHeadAttention(d_model, n_heads, n_kv_heads, causal, rope)
        self.xattn = MultiHeadAttention(d_model, n_heads) if cross else None
        self.ffn = keras.Sequential([layers.Dense(d_ff, activation="gelu"), layers.Dense(d_model)])
        self.ln1, self.ln2, self.ln3 = [layers.LayerNormalization(epsilon=1e-6) for _ in range(3)]
        self.drop = layers.Dropout(dropout)

    def call(self, x, context=None, training=False):
        if self.pre_ln:                                        # x + F(LN(x))
            x = x + self.drop(self.attn(self.ln1(x)), training=training)
            if self.cross: x = x + self.drop(self.xattn(self.ln3(x), context=context), training=training)
            x = x + self.drop(self.ffn(self.ln2(x)), training=training)
        else:                                                  # LN(x + F(x))
            x = self.ln1(x + self.drop(self.attn(x), training=training))
            if self.cross: x = self.ln3(x + self.drop(self.xattn(x, context=context), training=training))
            x = self.ln2(x + self.drop(self.ffn(x), training=training))
        return x

blk = TransformerBlock(64, 8, 256, n_kv_heads=2, causal=True, rope=True)
print("salida:", blk(tf.random.normal((2, 10, 64))).shape, "| parámetros:", blk.count_params())

In [ ]:
# ---- Verificaciones de corrección ----
mha = MultiHeadAttention(64, 8, causal=True); x = tf.random.normal((1, 12, 64)); o, a = mha(x, return_weights=True); a = a.numpy()
print("pesos suman 1 por fila:", np.allclose(a.sum(-1), 1.0, atol=1e-5))
print("causalidad (nada sobre la diagonal):", np.abs(np.triu(a[0, 0], 1)).max() < 1e-6)
x2 = x.numpy().copy(); x2[:, 8:] = np.random.randn(1, 4, 64); o2 = mha(tf.constant(x2))
print("posiciones 0..7 invariantes a cambios futuros:", np.allclose(o.numpy()[:, :8], o2.numpy()[:, :8], atol=1e-5))
for hk in [8, 4, 2, 1]:
    m = MultiHeadAttention(64, 8, n_kv_heads=hk); _ = m(x); print(f"n_kv_heads={hk}: parámetros = {m.count_params()}")

### Un modelo de lenguaje a nivel de caracteres (decoder-only)
Un **Transformer decoder-only** (como GPT) con RoPE y Pre-LN. Aquí se ve el **teacher forcing**: la entrada es la secuencia real y el objetivo es esa misma secuencia **desplazada una posición**.

In [ ]:
# ---- Datos: texto sintético con gramática simple ----
subj = ["el gato", "la niña", "un perro", "el profesor", "la estudiante"]; verb = ["duerme", "come", "estudia", "corre", "lee"]; adv = ["rápido", "despacio", "mucho", "poco", "bien"]
rng = np.random.default_rng(0)
text = " . ".join(f"{rng.choice(subj)} {rng.choice(verb)} {rng.choice(adv)}" for _ in range(6000)) + " ."
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = np.array([stoi[c] for c in text], dtype="int32"); V = len(chars); ctx = 48
def make_ds(arr, ctx, bs=128):
    xs = np.stack([arr[i:i + ctx] for i in range(0, len(arr) - ctx - 1, 3)]); ys = np.stack([arr[i + 1:i + ctx + 1] for i in range(0, len(arr) - ctx - 1, 3)])
    return tf.data.Dataset.from_tensor_slices((xs, ys)).shuffle(10000, seed=0).batch(bs).prefetch(2)
split = int(0.9 * len(data)); ds_tr, ds_va = make_ds(data[:split], ctx), make_ds(data[split:], ctx)
print("vocabulario:", V, "| caracteres:", len(text), "| muestra:", text[:70])

In [ ]:
class TinyGPT(keras.Model):
    def __init__(self, vocab, d_model=96, n_heads=4, n_layers=3, d_ff=256, n_kv_heads=None, pre_ln=True, rope=True):
        super().__init__()
        self.emb = layers.Embedding(vocab, d_model)
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff, n_kv_heads, causal=True, rope=rope, pre_ln=pre_ln) for _ in range(n_layers)]
        self.lnf = layers.LayerNormalization(epsilon=1e-6) if pre_ln else layers.Identity()
        self.head = layers.Dense(vocab)
    def call(self, ids, training=False):
        x = self.emb(ids)
        for b in self.blocks: x = b(x, training=training)
        return self.head(self.lnf(x))

gpt = TinyGPT(V, n_kv_heads=2)
gpt.compile(optimizer=keras.optimizers.AdamW(2e-3, weight_decay=0.01, clipnorm=1.0),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
hist = gpt.fit(ds_tr, validation_data=ds_va, epochs=6, verbose=2)
print(f"pérdida de validación final = {hist.history['val_loss'][-1]:.3f}  (entropía uniforme = {math.log(V):.3f})")

In [ ]:
# ---- Generación autoregresiva: greedy / temperatura / top-k / top-p ----
def generate(model, prompt, n_new=80, temperature=1.0, top_k=0, top_p=1.0, greedy=False, seed=0):
    rg = np.random.default_rng(seed); ids = [stoi[c] for c in prompt]
    for _ in range(n_new):
        logits = model(tf.constant([ids[-ctx:]]), training=False)[0, -1].numpy().astype("float64")
        if greedy: nxt = int(logits.argmax())
        else:
            logits = logits / max(temperature, 1e-6)
            if top_k > 0: logits[logits < np.sort(logits)[-top_k]] = -1e9
            p = softmax_np(logits)
            if top_p < 1.0:
                o = np.argsort(-p); cs = np.cumsum(p[o]); keep = o[: np.searchsorted(cs, top_p) + 1]
                m = np.zeros_like(p); m[keep] = p[keep]; p = m / m.sum()
            nxt = int(rg.choice(len(p), p=p))
        ids.append(nxt)
    return "".join(itos[i] for i in ids)

prompt = "el gato "
print("greedy   :", generate(gpt, prompt, greedy=True)); print("T=0.7    :", generate(gpt, prompt, temperature=0.7))
print("top-k=5  :", generate(gpt, prompt, top_k=5));    print("top-p=0.9:", generate(gpt, prompt, top_p=0.9))

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque D (nueva sección tras tu <i>## Transformer</i>).
</div>

---
## E. Estabilidad del entrenamiento y profundidad

### Teorema 5 (Colapso de rango de la atención pura; Dong, Cordonnier y Loukas, 2021) — 🎓 Avanzado
Definimos $\operatorname{res}(X)=\|X-\mathbf{1}x^\top\|$ con $x=\tfrac1nX^\top\mathbf{1}$ (distancia a la subvariedad de matrices con filas idénticas).

> **Teorema 5 (resumen).** En una red de $L$ capas de **autoatención pura** (sin residuales, MLP ni LayerNorm) con $H$ cabezas por capa y normas de pesos acotadas, existen constantes de las cotas de pesos y de la softmax tales que $\operatorname{res}(\mathrm{SAN}(X))$ está acotada por $C^{(3^{L}-1)/2}\operatorname{res}(X)^{3^{L}}$: **los tokens convergen a un único vector (rango 1) a velocidad doblemente exponencial en $L$.** Las **residuales** y los **MLP** rompen el colapso.

**Referencias:** Dong et al. (2021) [14] (constantes exactas: consultar el artículo); Noci et al. (2022) [15].

**Idea de la demostración.** (a) Descomponer la salida como suma de *caminos* sobre las matrices de atención de cada capa. (b) Cada matriz de atención es estocástica por filas y positiva, así que **contrae** $\operatorname{res}(\cdot)$ (Perron–Frobenius). (c) Como la matriz de atención depende a su vez de $X$ de forma polinómica, la contracción se *compone* con exponente $3$ por capa. $\qquad\blacksquare$ *(esquema; ver [14])*

In [ ]:
# ---- Verificación del Teorema 5: colapso según arquitectura ----
def res_np(X): return np.linalg.norm(X - X.mean(0, keepdims=True)) / np.linalg.norm(X)
def spec(W): return W / np.linalg.norm(W, 2)
def simulate(use_res, use_mlp, L=10, n=24, d=24, seed=0):
    r = np.random.default_rng(seed); X = r.normal(size=(n, d)); X = X / np.linalg.norm(X) * np.sqrt(n); out = [res_np(X)]
    for _ in range(L):
        WQ, WK, WV = [spec(r.normal(size=(d, d))) for _ in range(3)]
        Y = softmax_np((X @ WQ) @ (X @ WK).T / np.sqrt(d)) @ (X @ WV); X = X + Y if use_res else Y
        if use_mlp:
            W1, W2 = r.normal(size=(d, 4 * d)) / np.sqrt(d), r.normal(size=(4 * d, d)) / np.sqrt(4 * d); X = X + np.maximum(0, X @ W1) @ W2
        out.append(res_np(X))
    return np.array(out)
plt.figure(figsize=(7, 3.8))
for ur, um, lab in [(False, False, "atención pura"), (True, False, "+ residual"), (True, True, "+ residual + MLP")]:
    plt.semilogy(np.mean([simulate(ur, um, seed=s) for s in range(8)], 0) + 1e-18, "o-", label=lab)
plt.xlabel("capa $L$"); plt.ylabel("res$(X_L)$ relativo"); plt.title("T5: colapso de rango según la arquitectura"); plt.legend(); plt.show()
print("atención pura, res por capa:", np.array2string(simulate(False, False)[:5], precision=2))

### Teorema 6 (Gradientes en la inicialización: Pre-LN frente a Post-LN; Xiong et al., 2020) — 🎓 Avanzado
> **Teorema 6 (resumen, régimen de campo medio).** En la inicialización de un Transformer de $L$ capas y ancho $d$ grande:
> - **Post-LN** ($x_{l+1}=\mathrm{LN}(x_l+F_l(x_l))$): los gradientes cerca de la capa de salida son **grandes** (independientes de $L$ en orden), lo que exige *warm-up* del paso de aprendizaje.
> - **Pre-LN** ($x_{l+1}=x_l+F_l(\mathrm{LN}(x_l))$): los gradientes están **bien condicionados** y decrecen con la profundidad; el *warm-up* puede eliminarse.

**Referencias:** Xiong et al. (2020) [16] (constantes exactas en el artículo); Liu et al. (2020) [17]; Wang et al. (2024) [18].

**Idea.** En Pre-LN el jacobiano es $\prod_{j\ge l}(I+\partial F_j(\mathrm{LN}(x_j))/\partial x_j)$ y la normalización **fija la escala de la entrada de $F_j$**, manteniendo cada factor cerca de $I$. En Post-LN la normalización actúa **después** de la suma y reescala *todo* el flujo residual (incluida la identidad), así que los factores no están cerca de $I$. $\qquad\blacksquare$ *(esquema; ver [16])*

> **Advertencia.** Los enunciados de campo medio son **asintóticos**; la verificación siguiente es un **modelo simplificado** que reproduce la tendencia, no las constantes.

In [ ]:
# ---- Verificación (tendencia) del Teorema 6 ----
def ln_np(x): return (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-9)
def grad_profile(arch, L=48, d=64, seed=0, probe=(0, 8, 16, 24, 32, 40, 47)):
    r = np.random.default_rng(seed); Ws = [r.normal(size=(d, d)) * np.sqrt(2 / d) for _ in range(L)]; c, x0 = r.normal(size=d), r.normal(size=d)
    def layer(W, x):
        f = np.maximum(0, W @ (ln_np(x) if arch == "pre" else x)); return x + f if arch == "pre" else ln_np(x + f)
    def loss_from(l, xin):
        x = xin
        for W in Ws[l:]: x = layer(W, x)
        return c @ (ln_np(x) if arch == "pre" else x)
    xs = [x0]
    for W in Ws: xs.append(layer(W, xs[-1]))
    out = []
    for l in probe:
        g = np.zeros(d)
        for i in range(d):
            e = np.zeros(d); e[i] = 1e-5; g[i] = (loss_from(l, xs[l] + e) - loss_from(l, xs[l] - e)) / 2e-5
        out.append(np.linalg.norm(g))
    return np.array(out)
probe = (0, 8, 16, 24, 32, 40, 47)
pre, post = np.mean([grad_profile("pre", seed=s) for s in range(5)], 0), np.mean([grad_profile("post", seed=s) for s in range(5)], 0)
plt.figure(figsize=(6.5, 3.6)); plt.semilogy(probe, pre, "o-", label="Pre-LN"); plt.semilogy(probe, post, "s-", label="Post-LN")
plt.xlabel("capa $l$"); plt.ylabel(r"$\|\partial L/\partial x_l\|$"); plt.title("T6: perfil de gradientes en la inicialización"); plt.legend(); plt.show()
print(f"razón g(capa 0)/g(capa 47):  Pre-LN = {pre[0]/pre[-1]:.1f}  |  Post-LN = {post[0]/post[-1]:.1f}")
print(f"magnitud absoluta en capa 0: Pre-LN = {pre[0]:.1f}  |  Post-LN = {post[0]:.1f}")

In [ ]:
# ---- Experimento en TF: Pre-LN vs Post-LN con LR alto y SIN warm-up (reporte lo que obtenga) ----
def train_variant(pre_ln, lr, epochs=3, seed=0):
    tf.random.set_seed(seed)
    m = TinyGPT(V, d_model=64, n_heads=4, n_layers=6, d_ff=192, pre_ln=pre_ln, rope=True)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    return m.fit(ds_tr, validation_data=ds_va, epochs=epochs, verbose=0).history["val_loss"]
for lr in [1e-3, 5e-3]:
    for pre in [True, False]:
        print(f"lr={lr:.0e}  {'Pre-LN ' if pre else 'Post-LN'}  val_loss por época: {np.round(train_variant(pre, lr), 3)}")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque E.
</div>

---
## F. Atención eficiente e inferencia
La atención estándar materializa $S\in\mathbb{R}^{n\times n}$: memoria $O(n^2)$. En GPU el cuello de botella son los **accesos a memoria** (HBM↔SRAM), no los FLOPs. **FlashAttention** [19] calcula atención **exacta** por bloques.

### Teorema 7 (Softmax en línea y exactitud de la atención por bloques) — 🎓 Avanzado
Para $x\in\mathbb{R}^N$ recorrido por bloques $B_1,\dots,B_T$ mantenemos el máximo acumulado $m$ y la suma $\ell$:
$$m^{(t)}=\max\big(m^{(t-1)},\max x_{B_t}\big),\qquad \ell^{(t)}=e^{m^{(t-1)}-m^{(t)}}\ell^{(t-1)}+\sum_{i\in B_t}e^{x_i-m^{(t)}}.$$

> **Teorema 7.** (i) Al final, $m=\max_ix_i$ y $\ell=\sum_ie^{x_i-m}$; luego $\operatorname{softmax}(x)_i=e^{x_i-m}/\ell$ **exactamente**, en un solo recorrido. (ii) Para la atención, con la salida **sin normalizar** $O^{(t)}=e^{m^{(t-1)}-m^{(t)}}O^{(t-1)}+\sum_{j\in B_t}e^{s_j-m^{(t)}}v_j$ se cumple $O^{(T)}/\ell^{(T)}=\operatorname{softmax}(s)V$ **exactamente**. La memoria adicional es $O(n)$ y, con *tiling* en SRAM, los accesos a HBM bajan de $\Theta(nd+n^2)$ a $\Theta(n^2d^2/M)$ ($M$: tamaño de SRAM) [19].

**Referencias:** Milakov & Gimelshein (2018) [20]; Rabe & Staats (2021) [21]; Dao et al. (2022) [19]; Dao (2023) [22].

#### Demostración
Por inducción. Sea $\ell^{(t)}_\star=\sum_{i\in B_{1:t}}e^{x_i-m^{(t)}}$. Base $t=1$: inmediata. Paso: por hipótesis $\ell^{(t-1)}=\sum_{i\in B_{1:t-1}}e^{x_i-m^{(t-1)}}$; multiplicando por $e^{m^{(t-1)}-m^{(t)}}$ se obtiene $\sum_{i\in B_{1:t-1}}e^{x_i-m^{(t)}}$ y sumando el nuevo bloque queda $\ell^{(t)}=\sum_{i\in B_{1:t}}e^{x_i-m^{(t)}}$. Para (ii): $e^{s_j-m^{(t)}}=e^{m^{(t-1)}-m^{(t)}}e^{s_j-m^{(t-1)}}$, la misma recurrencia vale para el numerador; así $O^{(T)}=\sum_je^{s_j-m}v_j$ y $O^{(T)}/\ell^{(T)}=\operatorname{softmax}(s)V$. $\qquad\blacksquare$

*Observación.* FlashAttention da **los mismos números** (salvo redondeo); difiere de Linformer/Performer/atención lineal [23–25], que **sí aproximan** a cambio de coste $O(n)$.

In [ ]:
# ---- Verificación del Teorema 7 ----
def online_softmax_stats(x, block=64):
    m, l = -np.inf, 0.0
    for i in range(0, len(x), block):
        xb = x[i:i + block]; m_new = max(m, xb.max()); l = l * np.exp(m - m_new) + np.exp(xb - m_new).sum(); m = m_new
    return m, l
x = np.random.randn(5000) * 40
m, l = online_softmax_stats(x)
print(f"(i) m == max(x): {m == x.max()} | |ℓ − Σexp(x−max)| = {abs(l - np.exp(x - x.max()).sum()):.2e}")

def flash_attention_tf(Q, K, V, block=64, causal=False):
    n, d = Q.shape; m = tf.fill((n,), -np.inf); l = tf.zeros((n,)); O = tf.zeros((n, d))
    for j in range(0, n, block):
        Kj, Vj = K[j:j + block], V[j:j + block]
        S = tf.matmul(Q, Kj, transpose_b=True) / math.sqrt(d)
        if causal:
            qi = tf.range(n)[:, None]; kj = tf.range(j, j + Kj.shape[0])[None]; S = tf.where(kj <= qi, S, -1e9)
        m_new = tf.maximum(m, tf.reduce_max(S, axis=1)); P = tf.exp(S - m_new[:, None]); alpha = tf.exp(m - m_new)
        l = alpha * l + tf.reduce_sum(P, axis=1); O = alpha[:, None] * O + tf.matmul(P, Vj); m = m_new
    return O / l[:, None]

n, d = 512, 32; Q, K, Vv = [tf.random.normal((n, d)) for _ in range(3)]
exact = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / math.sqrt(d), axis=-1) @ Vv
for blk in [16, 64, 256]: print(f"(ii) bloque={blk:3d}: máx |Flash − exacta| = {tf.reduce_max(tf.abs(flash_attention_tf(Q, K, Vv, blk) - exact)).numpy():.2e}")
mask = tf.linalg.band_part(tf.ones((n, n)), -1, 0)
exact_c = tf.nn.softmax(tf.matmul(Q, K, transpose_b=True) / math.sqrt(d) + (1 - mask) * -1e9, axis=-1) @ Vv
print(f"causal: máx |Flash − exacta| = {tf.reduce_max(tf.abs(flash_attention_tf(Q, K, Vv, 64, causal=True) - exact_c)).numpy():.2e}")

ns = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])
plt.figure(figsize=(6.3, 3.5)); plt.loglog(ns, ns.astype(float)**2 * 4 / 2**20, "o-", label="estándar: $n^2$"); plt.loglog(ns, ns.astype(float) * 128 * 4 / 2**20, "s-", label="por bloques: $n\\cdot B$")
plt.xlabel("longitud $n$"); plt.ylabel("MiB por cabeza (fp32)"); plt.title("Memoria de la matriz de atención"); plt.legend(); plt.show()

### KV-cache, MQA y GQA
En generación, al producir el token $t$ se **reutilizan** claves y valores de los tokens previos. El **KV-cache** reduce el coste por token de $O(t\,d^2)$ a $O(d^2+t\,d)$, pero consume memoria:
$$\text{Mem}_{KV}=2\cdot L\cdot n\cdot h_{kv}\cdot d_h\cdot b\quad\text{(bytes)}.$$
**MHA:** $h_{kv}=h$. **MQA** (Shazeer 2019 [26]): $h_{kv}=1$. **GQA** (Ainslie et al. 2023 [13]): $1<h_{kv}<h$. Como la generación está limitada por **ancho de banda de memoria**, reducir $h_{kv}$ acelera la inferencia.

In [ ]:
def kv_gib(L, n, d_head, h_kv, bytes_=2, batch=1): return 2 * L * n * h_kv * d_head * bytes_ * batch / 2**30
print("KV-cache (L=32, d_head=128, n=8192, fp16, batch=1):")
for nm, hk in [("MHA  h_kv=32", 32), ("GQA  h_kv=8", 8), ("MQA  h_kv=1", 1)]: print(f"  {nm:14s}: {kv_gib(32, 8192, 128, hk):6.2f} GiB")

class CachedAttn(layers.Layer):
    def __init__(self, d, h, **kw):
        super().__init__(**kw); self.h, self.dh = h, d // h; self.wq, self.wk, self.wv, self.wo = [layers.Dense(d, use_bias=False) for _ in range(4)]
    def heads(self, x): B, n = tf.shape(x)[0], tf.shape(x)[1]; return tf.transpose(tf.reshape(x, (B, n, self.h, self.dh)), (0, 2, 1, 3))
    def full(self, x):
        q, k, v = self.heads(self.wq(x)), self.heads(self.wk(x)), self.heads(self.wv(x)); n = tf.shape(x)[1]
        s = tf.matmul(q, k, transpose_b=True) / math.sqrt(self.dh) + (1 - tf.linalg.band_part(tf.ones((n, n)), -1, 0)) * -1e9
        return self.wo(tf.reshape(tf.transpose(tf.matmul(tf.nn.softmax(s), v), (0, 2, 1, 3)), tf.shape(x)))
    def step(self, x_t, cache):
        q = self.heads(self.wq(x_t)); k_t, v_t = self.heads(self.wk(x_t)), self.heads(self.wv(x_t))
        K = k_t if cache is None else tf.concat([cache[0], k_t], 2); Vc = v_t if cache is None else tf.concat([cache[1], v_t], 2)
        o = tf.matmul(tf.nn.softmax(tf.matmul(q, K, transpose_b=True) / math.sqrt(self.dh)), Vc)
        return self.wo(tf.reshape(tf.transpose(o, (0, 2, 1, 3)), tf.shape(x_t))), (K, Vc)

att = CachedAttn(64, 4); x = tf.random.normal((1, 20, 64)); full = att.full(x); cache = None; outs = []
for t in range(20): o, cache = att.step(x[:, t:t + 1], cache); outs.append(o)
print("KV-cache == recomputar todo:", np.allclose(tf.concat(outs, 1).numpy(), full.numpy(), atol=1e-5))
T = np.arange(1, 2049); print(f"pares query-key al generar 2048 tokens: sin cache {int(np.cumsum(T*(T+1)/2)[-1]):,} | con cache {int(np.cumsum(T)[-1]):,}")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque F.
</div>

---
## G. Leyes de escalamiento
**Definición (ley tipo Chinchilla).** La pérdida se modela como
$$\boxed{L(N,D)=E+\frac{A}{N^{\alpha}}+\frac{B}{D^{\beta}}},\qquad C\approx6ND,$$
con $E$ la entropía irreducible, $N$ parámetros, $D$ tokens y $C$ FLOPs de entrenamiento (6 por parámetro y token: 2 *forward* + 4 *backward*).

### Teorema 8 (Asignación óptima de cómputo; Hoffmann et al., 2022)
> **Teorema 8.** Minimizando $L(N,D)$ sujeto a $6ND=C$ (con $\alpha,\beta>0$), el óptimo es
> $$N^\star(C)=G\Big(\tfrac C6\Big)^{\frac{\beta}{\alpha+\beta}},\qquad D^\star(C)=G^{-1}\Big(\tfrac C6\Big)^{\frac{\alpha}{\alpha+\beta}},\qquad G=\Big(\tfrac{\alpha A}{\beta B}\Big)^{\frac1{\alpha+\beta}}.$$
> Luego $N^\star\propto C^{a}$, $D^\star\propto C^{b}$ con $a=\tfrac{\beta}{\alpha+\beta}$, $b=\tfrac{\alpha}{\alpha+\beta}$ y **$a+b=1$**. Con los valores ajustados por Hoffmann et al. ($\alpha\approx0.34,\ \beta\approx0.28$): $a\approx0.45,\ b\approx0.55$, es decir, **modelo y datos deben crecer casi a la par**. Kaplan et al. [27] habían estimado un exponente de $N$ mayor (favoreciendo modelos más grandes con menos datos); la diferencia se atribuye a la metodología de ajuste.

**Referencias:** Kaplan et al. (2020) [27]; Hoffmann et al. (2022) [28]; Besiroglu et al. (2024) [29] (réplica que cuestiona la precisión del ajuste original); Muennighoff et al. (2023) [30]; Sardana et al. (2024) [31] (incluye el coste de **inferencia**).

#### Demostración
Con $D=C/(6N)$: $f(N)=E+AN^{-\alpha}+B(C/6)^{-\beta}N^{\beta}$. Derivando, $-\alpha AN^{-\alpha-1}+\beta B(C/6)^{-\beta}N^{\beta-1}=0\Rightarrow N^{\alpha+\beta}=\frac{\alpha A}{\beta B}(C/6)^{\beta}$, de donde $N^\star$. $f$ es **convexa en $\log N$** (suma de exponenciales positivas en $\log N$), así que el punto crítico es el mínimo global. Luego $D^\star=C/(6N^\star)\propto C^{\alpha/(\alpha+\beta)}$. $\qquad\blacksquare$

In [ ]:
# ---- Verificación del Teorema 8: fórmula cerrada vs. minimización numérica ----
E_, A_, B_, al, be = 1.69, 406.4, 410.7, 0.34, 0.28            # Hoffmann et al. 2022 (ajuste publicado)
G = (al * A_ / (be * B_)) ** (1 / (al + be))
def closed(C): return G * (C / 6) ** (be / (al + be)), (C / 6) ** (al / (al + be)) / G
def numeric(C):
    Ns = np.logspace(5, 14, 200000); Ds = C / (6 * Ns); L = E_ + A_ / Ns**al + B_ / Ds**be; i = L.argmin(); return Ns[i], Ds[i], L[i]
print(f"exponentes: N* ∝ C^{be/(al+be):.3f}   D* ∝ C^{al/(al+be):.3f}   (suman {be/(al+be)+al/(al+be):.1f})")
print(f"{'C':>8} {'N* cerrada':>12} {'N* numérica':>12} {'err. rel.':>10} {'D/N':>6} {'L*':>6}")
for C in [1e21, 1e22, 1e23, 1e24]:
    Nc, Dc = closed(C); Nn, Dn, Ln = numeric(C); print(f"{C:8.0e} {Nc:12.3e} {Nn:12.3e} {abs(Nc-Nn)/Nn:10.1e} {Dc/Nc:6.1f} {Ln:6.3f}")
Ns, Ds = np.logspace(8, 11.5, 200), np.logspace(9, 13, 200); NN, DD = np.meshgrid(Ns, Ds); LL = E_ + A_ / NN**al + B_ / DD**be
plt.figure(figsize=(6.5, 4.5)); cs = plt.contourf(np.log10(NN), np.log10(DD), LL, 25, cmap="viridis"); plt.colorbar(cs, label="$L(N,D)$")
fr = np.array([closed(c) for c in np.logspace(20, 25, 50)]); plt.plot(np.log10(fr[:, 0]), np.log10(fr[:, 1]), "r-", lw=2, label="óptimo $N^\\star(C),D^\\star(C)$")
plt.xlabel("$\\log_{10}N$"); plt.ylabel("$\\log_{10}D$"); plt.title("T8: frontera compute-óptima"); plt.legend(); plt.show()

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque G.
</div>

---
## H. Expresividad y límites teóricos
Dos preguntas complementarias: ¿**qué** puede aproximar un Transformer? y ¿**qué no** puede computar bajo supuestos realistas?

### Teorema 9 (Aproximación universal; Yun et al., 2020) — 🎓 Avanzado
> **Teorema 9.** Sea $1\le p<\infty$, $\varepsilon>0$ y $\mathcal{D}\subset\mathbb{R}^{d\times n}$ **compacto**. Con tamaños de cabeza y de FFN **fijos** (independientes de $n$ y $d$):
> 1. **(Equivariantes)** Para toda $f$ continua y **equivariante a permutaciones** existe un Transformer $g$ con $d_p(f,g)=\big(\int_{\mathcal{D}}\|f(X)-g(X)\|_p^p\,dX\big)^{1/p}<\varepsilon$.
> 2. **(Con codificación posicional)** Para toda $f$ continua (sin restricción de simetría) existe un Transformer con **codificación posicional entrenable** con $d_p(f,g)<\varepsilon$.
>
> **Salvedades.** (a) Aproximación en norma $L^p$, no uniforme. (b) Es un resultado de **existencia**: el número de capas puede ser enorme y **no** dice nada sobre aprendibilidad por descenso de gradiente. (c) Las **FFN son imprescindibles** (consistente con T5). (d) Una capa de atención aislada **no** es aproximador universal.

**Referencias:** Yun et al. (2020) [3]; Edelman et al. (2022) [32]; Jiang & Li (2024) [33].

**Idea de la demostración.** (1) Aproximar $f$ por una función constante a trozos sobre una malla y usar tres etapas: **(i)** capas FFN que *cuantizan* la entrada; **(ii)** capas de autoatención con **hardmax** (aproximable con softmax de baja temperatura) que construyen un *mapa contextual*, asignando un identificador único a cada (secuencia, token) —la atención puede calcular máximos/mínimos sobre los tokens—; **(iii)** FFN que mapean el identificador al valor de $f$. (2) La codificación posicional **rompe la equivarianza** (T1) y se aplica (1) a la función extendida. $\qquad\blacksquare$ *(esquema; ver [3])*

In [ ]:
# ---- Ilustración de T1 + T9: equivariante vs. no equivariante, con y sin codificación posicional ----
rng = np.random.default_rng(0); n_, d_ = 8, 4
def make(task, N=6000):
    X = rng.normal(size=(N, n_, d_)).astype("float32"); Y = (X - X.mean(1, keepdims=True)) if task == "center" else X[:, ::-1, :]
    return X, Y.astype("float32")
def build(use_pe):
    inp = keras.Input((n_, d_)); h = layers.Dense(64)(inp)
    if use_pe: h = h + layers.Embedding(n_, 64)(tf.range(n_))[None]
    for _ in range(2): h = TransformerBlock(64, 4, 128, pre_ln=True, dropout=0.0)(h)
    return keras.Model(inp, layers.Dense(d_)(h))
res = {}
for task in ["center", "reverse"]:
    X, Y = make(task); Xt, Yt = make(task, 1000)
    for use_pe in [False, True]:
        m = build(use_pe); m.compile(keras.optimizers.Adam(2e-3), "mse"); m.fit(X, Y, epochs=25, batch_size=128, verbose=0); res[(task, use_pe)] = m.evaluate(Xt, Yt, verbose=0)
print(f"{'tarea':10s} {'sin PE':>10s} {'con PE':>10s}   (MSE de prueba; varianza de la salida ≈ 1.0)")
for task in ["center", "reverse"]: print(f"{task:10s} {res[(task, False)]:10.4f} {res[(task, True)]:10.4f}")
print("Lectura: 'reverse' no es equivariante, así que SIN PE no puede aprenderse (T1); con PE sí (T9.2). 'center' es equivariante y no necesita PE.")

### Teorema 10 (Límites de expresividad con precisión finita; Hahn 2020, Merrill y Sabharwal 2023–24) — 🎓 Avanzado
Cada parte vale **bajo supuestos concretos**; léalas con sus hipótesis.

> **Teorema 10 (resumen en tres partes).**
> **(a) [Hahn, 2020]** Para Transformers con **número fijo de capas y cabezas** y *hard attention*, el lenguaje $\mathrm{PARITY}=\{w\in\{0,1\}^*:\#_1(w)\ \text{par}\}$ **no es reconocible**; para *soft attention*, la entropía cruzada por token sobre PARITY o Dyck-2 se acerca a la de un predictor trivial cuando $n\to\infty$ (forma cuantitativa exacta en [34]; aquí sólo se afirma la tendencia).
> **(b) [Merrill y Sabharwal, 2023]** Todo Transformer de **profundidad constante** y **precisión logarítmica** ($c\log n$ bits) puede simularse por circuitos **$\mathsf{TC}^0$** uniformes en espacio logarítmico. Por tanto, **si** $\mathsf{TC}^0\neq\mathsf{NC}^1$ (conjetura estándar), esos Transformers **no** resuelven problemas $\mathsf{NC}^1$-completos; ejemplos citados en [35]: la **composición de permutaciones de $S_5$** y la **evaluación de fórmulas booleanas**. *(No debe concluirse nada general sobre otros problemas sin analizar su clase de complejidad.)*
> **(c) [Merrill y Sabharwal, 2024; Li et al., 2024]** Estas limitaciones **se relajan** con **chain of thought**: con $\mathrm{poly}(n)$ tokens intermedios el decoder-only puede resolver problemas de $\mathsf{P}$ (bajo condiciones de uniformidad); el CoT aporta **cómputo serial** que la profundidad constante no tiene.
>
> **Salvedades.** (i) (a) *no* implica que PARITY sea imposible en la práctica: **Chiang y Cholak (2022)** [36] dan construcciones explícitas que la reconocen para todas las longitudes. (ii) (b) exige **precisión logarítmica**. (iii) Las cotas $\mathsf{TC}^0$ son **superiores** de expresividad y **condicionales** a una conjetura de complejidad.

**Referencias:** Hahn (2020) [34]; Merrill y Sabharwal (2023) [35, 37]; Chiang y Cholak (2022) [36]; Merrill y Sabharwal (2024) [38]; Li et al. (2024) [39]; encuesta: Strobl et al. (2024) [40].

**Idea de la demostración de (b).** Se simula cada componente por un circuito de umbral de profundidad constante: **(i)** el producto punto de vectores de $c\log n$ bits está en $\mathsf{TC}^0$; **(ii)** la **suma iterada** $\sum_je^{s_j}v_j$ de $n$ términos está en $\mathsf{TC}^0$; **(iii)** la división de la softmax también está en $\mathsf{TC}^0$. La profundidad total del Transformer es constante, así que la composición lo es. $\qquad\blacksquare$ *(esquema; ver [35])*

In [ ]:
# ---- Ilustración de T10: PARITY. Se MIDE (no se afirma) la precisión al crecer n con profundidad y cabezas fijas ----
rng = np.random.default_rng(1)
def parity_data(N, n): X = rng.integers(0, 2, size=(N, n)).astype("int32"); return X, (X.sum(1) % 2).astype("int32")
def build_cls(n, d=64, layers_=2):
    inp = keras.Input((n,), dtype="int32"); h = layers.Embedding(2, d)(inp) + layers.Embedding(n, d)(tf.range(n))[None]
    for _ in range(layers_): h = TransformerBlock(d, 4, 4 * d, pre_ln=True, dropout=0.0)(h)
    return keras.Model(inp, layers.Dense(2)(layers.GlobalAveragePooling1D()(h)))
acc_by_n = {}
for n_try in [6, 10, 14, 18]:
    Xa, ya = parity_data(30000, n_try); Xb, yb = parity_data(3000, n_try)
    mm = build_cls(n_try); mm.compile(keras.optimizers.Adam(1e-3), keras.losses.SparseCategoricalCrossentropy(from_logits=True), ["accuracy"])
    mm.fit(Xa, ya, epochs=30, batch_size=256, verbose=0); acc_by_n[n_try] = mm.evaluate(Xb, yb, verbose=0)[1]
    print(f"n={n_try:2d}  precisión de validación = {acc_by_n[n_try]:.3f}   (azar = 0.5)")
plt.figure(figsize=(5.5, 3)); plt.plot(list(acc_by_n), list(acc_by_n.values()), "o-"); plt.axhline(0.5, color="gray", ls="--", label="azar")
plt.xlabel("longitud $n$"); plt.ylabel("precisión"); plt.title("PARITY con profundidad y cabezas fijas"); plt.legend(); plt.show()

# Chain-of-thought: la paridad PREFIJO p_t = p_{t-1} XOR x_t es una recurrencia (cómputo serial) que un decoder causal aprende por token
def parity_cot(N, n): X = rng.integers(0, 2, size=(N, n)).astype("int32"); return X, (np.cumsum(X, 1) % 2).astype("int32")
n_tr = 12; Xc, Pc = parity_cot(30000, n_tr)
inp = keras.Input((n_tr,), dtype="int32"); h = layers.Embedding(2, 64)(inp) + layers.Embedding(n_tr, 64)(tf.range(n_tr))[None]
for _ in range(2): h = TransformerBlock(64, 4, 256, causal=True, pre_ln=True, dropout=0.0)(h)
mc = keras.Model(inp, layers.Dense(2)(h)); mc.compile(keras.optimizers.Adam(1e-3), keras.losses.SparseCategoricalCrossentropy(from_logits=True), ["accuracy"])
mc.fit(Xc, Pc, epochs=25, batch_size=256, verbose=0); Xcv, Pcv = parity_cot(3000, n_tr)
print(f"paridad PREFIJO (supervisión tipo scratchpad): precisión por token = {mc.evaluate(Xcv, Pcv, verbose=0)[1]:.3f}")
print("Reporte lo obtenido. La teoría predice dificultad de GENERALIZAR con profundidad fija (Hahn), y que el CoT convierte el problema serial en pasos locales (T10c).")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque H.
</div>

---
## I. Interpretabilidad mecanicista: circuitos e induction heads
Elhage et al. [41] formalizan el Transformer con una vista **lineal**.

**Flujo residual.** Cada capa **lee** del flujo residual $x\in\mathbb{R}^{d}$ y **escribe** una actualización aditiva: $x\leftarrow x+\sum_h\text{head}_h(x)+\text{MLP}(x)$.

**Descomposición QK/OV.** Una cabeza se factoriza en dos circuitos independientes:
$$\underbrace{A^{(h)}=\operatorname{softmax}\!\Big(\tfrac{XW_Q^{(h)}W_K^{(h)\top}X^\top}{\sqrt{d_k}}\Big)}_{\text{circuito QK: \emph{dónde} atender}}\qquad \underbrace{W_{OV}^{(h)}=W_V^{(h)}W_O^{(h)}}_{\text{circuito OV: \emph{qué} escribir}},$$
y la salida de la cabeza es $A^{(h)}XW_{OV}^{(h)}$. Sólo importan los **productos** $W_QW_K^\top$ y $W_VW_O$ (simetría $W_Q\to W_QR,\ W_K\to W_KR^{-\top}$).

### Proposición (Cabezas de inducción)
> Una cabeza de inducción es un circuito de **dos capas**: (1) una *previous-token head* copia a cada posición $j$ la identidad de $x_{j-1}$ a un subespacio del flujo residual; (2) una *induction head* usa como **clave** ese contenido y como **consulta** el token actual $x_t$, atiende a toda posición $j$ cuyo token previo coincide con $x_t$ y copia (circuito OV) $x_j$ al logit de salida. Implementa $[A][B]\cdots[A]\mapsto[B]$.

**Evidencia:** Olsson et al. (2022) [42] muestran que la formación de induction heads coincide con un **salto brusco** en la capacidad de **aprendizaje en contexto**. *Nota:* la evidencia causal es más fuerte en modelos pequeños; en grandes es **correlacional** (los propios autores lo señalan).

In [ ]:
# ---- Induction head construida a mano ----
rng = np.random.default_rng(5); Vv_, T_, d_ = 64, 40, 128
emb = rng.normal(size=(Vv_, d_)); emb /= np.linalg.norm(emb, axis=1, keepdims=True)
def induction_predict(tokens, t, sharp=30.0):
    X = emb[tokens]; prev = np.zeros_like(X); prev[1:] = X[:-1]          # capa 1: previous-token head
    sc = (X[t] @ prev.T) * sharp; sc[t + 1:] = -1e9; sc[0] = -1e9        # capa 2: QK (query=token actual, key=token previo de j) + causalidad
    w = softmax_np(sc); return int(((w @ X) @ emb.T).argmax()), w        # OV: copia el token en j
ok = 0; N = 300
for s in range(N):
    r = np.random.default_rng(100 + s); tk = r.permutation(Vv_)[:T_].copy(); i, j = r.integers(1, 15), r.integers(20, T_ - 1); tk[j] = tk[i]
    ok += (induction_predict(tk, j)[0] == tk[i + 1])
print(f"precisión de la induction head: {ok/N:.3f} sobre {N} secuencias")
tk = np.random.default_rng(100).permutation(Vv_)[:T_].copy(); tk[25] = tk[5]; _, w = induction_predict(tk, 25)
plt.figure(figsize=(7, 2.6)); plt.bar(range(T_), w); plt.axvline(6, color="r", ls="--", label="posición tras A (=6)"); plt.title("Atención de la induction head desde la posición 25"); plt.legend(); plt.show()

In [ ]:
# ---- Verificación de la factorización QK/OV y de la simetría de gauge ----
d, h, dk, n = 32, 4, 8, 10
X = tf.random.normal((n, d)); WQ, WK, WV = [tf.random.normal((h, d, dk)) / math.sqrt(d) for _ in range(3)]; WO = tf.random.normal((h, dk, d)) / math.sqrt(dk)
heads = [tf.nn.softmax((X @ WQ[i]) @ tf.transpose(X @ WK[i]) / math.sqrt(dk)) @ (X @ WV[i]) for i in range(h)]
out_concat = tf.concat(heads, -1) @ tf.reshape(WO, (h * dk, d))
out_sum = tf.add_n([tf.nn.softmax((X @ WQ[i]) @ tf.transpose(X @ WK[i]) / math.sqrt(dk)) @ X @ (WV[i] @ WO[i]) for i in range(h)])
print(f"||concat(heads)·W^O − Σ_h A_h X W_OV,h|| = {tf.norm(out_concat - out_sum).numpy():.2e}   (≈ 0: cabezas aditivas e independientes)")
R = tf.random.normal((dk, dk)) + 3 * tf.eye(dk); Rinv = tf.linalg.inv(R)
s1 = (X @ WQ[0]) @ tf.transpose(X @ WK[0]); s2 = (X @ (WQ[0] @ R)) @ tf.transpose(X @ (WK[0] @ tf.transpose(Rinv)))
print(f"invariancia de gauge (Q→QR, K→KR^-T): ||S − S'|| = {tf.norm(s1 - s2).numpy():.2e}")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque I.
</div>

---
## J. Aplicación en ciencia de datos: series de tiempo y datos tabulares
Los Transformers nacieron en NLP; su uso en **datos estructurados** es un debate abierto y muy relevante. Aquí el **escepticismo empírico** es central.

### ¿Ayudan los Transformers en series de tiempo?
- **Zeng et al. (2023)** [43]: un modelo **lineal de una capa** (*DLinear*) **superaba** a varios Transformers especializados en predicción a largo plazo. Argumento: la autoatención es **equivariante a permutaciones** (T1) y la codificación posicional recupera el orden de forma imperfecta, mientras que en series temporales *el orden lo es todo*.
- **Nie et al. (2023)** [44] respondieron con **PatchTST**: (i) **parches** (subseries) como *tokens*, conservando semántica local y reduciendo el coste de atención de $O(L^2)$ a $O((L/P)^2)$; (ii) **independencia de canales**.
- **Ansari et al. (2024)** [45] (*Chronos*): **tokenizan** los valores (escalado + cuantización) y entrenan un modelo de lenguaje con entropía cruzada: un **modelo fundacional** de series con pronóstico *zero-shot*.

**Conclusión pedagógica.** No hay ganador universal. La comparación honesta exige **líneas base fuertes** (naive estacional, regresión lineal, ARIMA/ETS) y validación temporal correcta (*rolling origin*, **nunca barajar**).

### Datos tabulares
**FT-Transformer** (Gorishniy et al., 2021) [46]: cada columna se *tokeniza* en un embedding y se aplica un Transformer con `[CLS]`. Es una línea base fuerte entre redes profundas, pero **los árboles potenciados siguen siendo muy competitivos** en tabulares de tamaño medio [47, 48].

In [ ]:
# ---- Series de tiempo: naive estacional vs. lineal vs. Transformer con parches (split TEMPORAL, sin barajar entre train y test) ----
rng = np.random.default_rng(0); T = 4000; t = np.arange(T)
series = (0.002 * t + 2 * np.sin(2 * np.pi * t / 24) + 1.0 * np.sin(2 * np.pi * t / 168) + 0.4 * rng.normal(size=T)).astype("float32")
L, H, P = 96, 24, 8
def windows(s, L, H, stride=2):
    idx = range(0, len(s) - L - H, stride); return np.stack([s[i:i + L] for i in idx]), np.stack([s[i + L:i + L + H] for i in idx])
cut = int(0.7 * T); Xtr, Ytr = windows(series[:cut], L, H); Xte, Yte = windows(series[cut:], L, H)
def revin(x): m = x.mean(1, keepdims=True); return x - m, m
mse_naive = np.mean((np.tile(Xte[:, -24:], (1, H // 24)) - Yte) ** 2)
xl, ml = revin(Xtr); xl_te, ml_te = revin(Xte)
lin = keras.Sequential([keras.Input((L,)), layers.Dense(H)]); lin.compile(keras.optimizers.Adam(1e-3), "mse"); lin.fit(xl, Ytr - ml, epochs=30, batch_size=64, verbose=0)
mse_lin = np.mean((lin.predict(xl_te, verbose=0) + ml_te - Yte) ** 2)
def build_patch(L, P, H, d=64, nl=2):
    npatch = L // P; inp = keras.Input((L,)); h = layers.Reshape((npatch, P))(inp); h = layers.Dense(d)(h) + layers.Embedding(npatch, d)(tf.range(npatch))[None]
    for _ in range(nl): h = TransformerBlock(d, 4, 2 * d, pre_ln=True, dropout=0.1)(h)
    return keras.Model(inp, layers.Dense(H)(layers.Flatten()(h)))
pt = build_patch(L, P, H); pt.compile(keras.optimizers.Adam(1e-3), "mse"); pt.fit(xl, Ytr - ml, epochs=30, batch_size=64, verbose=0, validation_split=0.1)
mse_pt = np.mean((pt.predict(xl_te, verbose=0) + ml_te - Yte) ** 2)
print(f"MSE de prueba (horizonte {H}): naive estacional = {mse_naive:.4f} | lineal = {mse_lin:.4f} | Transformer con parches = {mse_pt:.4f}")
print("(ruido irreducible ≈ 0.16). REPORTE el ranking que obtenga: la lección es comparar con líneas base fuertes.")

In [ ]:
# ---- FT-Transformer (tabulares) frente a Gradient Boosting ----
class FTTransformer(keras.Model):
    def __init__(self, n_num, cat_cards, d=32, n_heads=4, n_layers=2, n_out=1):
        super().__init__()
        self.num_w = self.add_weight(shape=(n_num, d), initializer="glorot_uniform", name="num_w"); self.num_b = self.add_weight(shape=(n_num, d), initializer="zeros", name="num_b")
        self.cat_emb = [layers.Embedding(c, d) for c in cat_cards]; self.cls = self.add_weight(shape=(1, 1, d), initializer="glorot_uniform", name="cls")
        self.blocks = [TransformerBlock(d, n_heads, 2 * d, pre_ln=True, dropout=0.1) for _ in range(n_layers)]; self.ln, self.head = layers.LayerNormalization(), layers.Dense(n_out)
    def call(self, inputs, training=False):
        xn, xc = inputs
        toks = [xn[:, :, None] * self.num_w[None] + self.num_b[None]] + [e(xc[:, i])[:, None, :] for i, e in enumerate(self.cat_emb)]
        h = tf.concat([tf.tile(self.cls, (tf.shape(xn)[0], 1, 1))] + toks, axis=1)
        for b in self.blocks: h = b(h, training=training)
        return self.head(self.ln(h[:, 0]))
rng = np.random.default_rng(0); N = 6000
xn = rng.normal(size=(N, 5)).astype("float32"); xc = rng.integers(0, 4, size=(N, 2)).astype("int32")
y = (np.sin(xn[:, 0]) + xn[:, 1] * (xc[:, 0] == 2) + 0.5 * xn[:, 2] ** 2 + 0.2 * rng.normal(size=N)).astype("float32")
tr, te = slice(0, 5000), slice(5000, N)
ft = FTTransformer(5, [4, 4]); ft.compile(keras.optimizers.AdamW(1e-3, weight_decay=1e-4), "mse"); ft.fit((xn[tr], xc[tr]), y[tr], epochs=40, batch_size=128, verbose=0)
from sklearn.ensemble import GradientBoostingRegressor
X_gb = np.hstack([xn, xc]); gb = GradientBoostingRegressor(random_state=0).fit(X_gb[tr], y[tr])
print(f"MSE prueba: FT-Transformer = {np.mean((ft.predict((xn[te], xc[te]), verbose=0)[:, 0] - y[te])**2):.4f} | Gradient Boosting = {np.mean((gb.predict(X_gb[te]) - y[te])**2):.4f}")
print("REPORTE ambos números: los árboles suelen ser una línea base muy fuerte en tabulares; compare siempre.")

<div style="border-left:5px solid #c0392b;background:#fdecea;padding:8px 12px;margin:6px 0">
<b>📍 Punto de inserción en tu notebook original:</b> después del bloque J.
</div>

---
## K. Mapa de frontera
Cinco temas que **completan el panorama**: definición, resultado clave, ejemplo mínimo y bibliografía. No se demuestran por completo; son puntos de partida para proyectos y tesis.

### K.1 Mixture of Experts (MoE)
Reemplazar la FFN densa por $E$ expertos y un **enrutador** que activa sólo los $k$ mejores por token:
$$y=\sum_{e\in\text{Top-}k(g(x))}\operatorname{softmax}(g(x))_e\,\mathrm{FFN}_e(x),\qquad g(x)=W_rx.$$
Los parámetros **totales** crecen con $E$ y el cómputo **activo** sólo con $k$. *Switch Transformer* [49] usa $k=1$ y una **pérdida auxiliar de balance de carga** $\mathcal{L}_{aux}=E\sum_ef_eP_e$ ($f_e$: fracción de tokens enrutados a $e$; $P_e$: probabilidad media del enrutador). *Mixtral 8×7B* [50]: $E=8$, $k=2$.

### K.2 Vision Transformer (ViT) y multimodal
**ViT** [51]: dividir la imagen en parches, aplanarlos, proyectarlos linealmente y tratarlos como *tokens* (+ posición + `[CLS]`). Con poco dato las CNN generalizan mejor (sesgo inductivo de localidad); con **mucho** dato ViT las iguala o supera. **CLIP** [52]: dos codificadores con pérdida **contrastiva simétrica** (InfoNCE) sobre un lote de $B$ pares:
$$\mathcal{L}=-\frac1{2B}\sum_{i=1}^{B}\Big[\log\frac{e^{s_{ii}/\tau}}{\sum_je^{s_{ij}/\tau}}+\log\frac{e^{s_{ii}/\tau}}{\sum_je^{s_{ji}/\tau}}\Big],\quad s_{ij}=\langle f(I_i),g(T_j)\rangle.$$

### K.3 LoRA y QLoRA
**LoRA** [53]: congelar $W_0\in\mathbb{R}^{d\times k}$ y entrenar una actualización de **rango bajo** $W=W_0+\tfrac{\alpha}{r}BA$, con $B\in\mathbb{R}^{d\times r}$, $A\in\mathbb{R}^{r\times k}$, $r\ll\min(d,k)$ y $B=0$ al inicio. Parámetros entrenables: $r(d+k)$ en vez de $dk$. Hipótesis subyacente: **baja dimensión intrínseca** del ajuste fino [54]. **QLoRA** [55] cuantiza $W_0$ a 4 bits (*NormalFloat*) y entrena adaptadores LoRA en 16 bits.

### K.4 Alineación: RLHF y DPO
**RLHF** [56, 57]: (1) ajuste supervisado; (2) **modelo de recompensa** con Bradley–Terry, $P(y_w\succ y_l)=\sigma(r(x,y_w)-r(x,y_l))$; (3) optimizar la política con RL (PPO) bajo penalización KL:
$$\max_\pi\ \mathbb{E}_{x,\,y\sim\pi}[r(x,y)]-\beta\,D_{KL}\big(\pi(\cdot|x)\,\|\,\pi_{ref}(\cdot|x)\big).$$
**DPO** [58]: el óptimo de ese problema es $\pi^\star\propto\pi_{ref}e^{r/\beta}$, de donde $r=\beta\log(\pi/\pi_{ref})+\text{cte}(x)$ y se **elimina el modelo de recompensa y el RL**:
$$\mathcal{L}_{DPO}=-\mathbb{E}\Big[\log\sigma\Big(\beta\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\beta\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\Big)\Big].$$
*(Conecta con el curso de VAE: optimización con penalización KL.)*

### K.5 Modelos de espacio de estados (S4, Mamba)
Un **SSM** lineal e invariante en el tiempo es $h'(t)=Ah(t)+Bx(t),\ y(t)=Ch(t)$; discretizado, $h_t=\bar Ah_{t-1}+\bar Bx_t$. Se evalúa como **recurrencia** (inferencia $O(1)$ por token) o como **convolución** (entrenamiento paralelo). **S4** [59] parametriza $A$ con estructura (HiPPO + NPLR). **Mamba** [60] hace los parámetros **dependientes de la entrada** (selectividad) con un *scan* paralelo consciente del hardware. **Mamba-2** [61] establece la **dualidad** SSM–atención. *Debate abierto:* los híbridos (Transformer + SSM) suelen superar a ambos puros; los SSM puros tienen dificultad en tareas de **recuperación exacta** (*copy*, *associative recall*), que la atención resuelve de forma natural (Bloque I).

In [ ]:
# ---- K.1: capa MoE top-k con pérdida de balance de carga ----
class MoE(layers.Layer):
    def __init__(self, d, d_ff, n_exp=4, k=1, **kw):
        super().__init__(**kw); self.E, self.k = n_exp, k; self.router = layers.Dense(n_exp, use_bias=False)
        self.experts = [keras.Sequential([layers.Dense(d_ff, activation="gelu"), layers.Dense(d)]) for _ in range(n_exp)]
    def call(self, x):
        probs = tf.nn.softmax(self.router(x), -1); topv, topi = tf.math.top_k(probs, self.k); gate = topv / tf.reduce_sum(topv, -1, keepdims=True)
        outs = tf.stack([e(x) for e in self.experts], -2)                       # versión densa (didáctica)
        sel = tf.reduce_sum(tf.one_hot(topi, self.E) * gate[..., None], -2); y = tf.reduce_sum(sel[..., None] * outs, -2)
        f = tf.reduce_mean(tf.reduce_sum(tf.one_hot(topi, self.E), -2), axis=[0, 1]) / self.k; P = tf.reduce_mean(probs, axis=[0, 1])
        self.add_loss(0.01 * self.E * tf.reduce_sum(f * P)); return y
moe = MoE(32, 64, n_exp=4, k=2); y = moe(tf.random.normal((2, 10, 32)))
print("salida MoE:", y.shape, "| pérdida auxiliar:", float(tf.add_n(moe.losses)), "| parámetros totales:", moe.count_params())

# ---- K.3: LoRA ----
class LoRADense(layers.Layer):
    def __init__(self, base, r=4, alpha=8, **kw): super().__init__(**kw); self.base, self.r, self.alpha = base, r, alpha; base.trainable = False
    def build(self, input_shape):
        self.A = self.add_weight(shape=(input_shape[-1], self.r), initializer="he_uniform", name="A"); self.B = self.add_weight(shape=(self.r, self.base.units), initializer="zeros", name="B")
    def call(self, x): return self.base(x) + (self.alpha / self.r) * (x @ self.A) @ self.B
base = layers.Dense(256); x = tf.random.normal((3, 128)); _ = base(x); lora = LoRADense(base, r=4)
print("LoRA con B=0 reproduce la capa base:", np.allclose(lora(x).numpy(), base(x).numpy(), atol=1e-6))
tr_p = sum(int(np.prod(v.shape)) for v in lora.trainable_variables); full = 128 * 256 + 256; print(f"entrenables LoRA = {tr_p} frente a {full} del ajuste completo ({100*tr_p/full:.1f} %)")

# ---- K.4: pérdida DPO ----
def dpo_loss(lp_w, lp_l, ref_w, ref_l, beta=0.1):
    margin = beta * ((lp_w - ref_w) - (lp_l - ref_l)); return -tf.reduce_mean(tf.math.log_sigmoid(margin)), margin
lw, ll, rw, rl = [tf.constant(v) for v in ([-10., -12., -9.], [-11., -11.5, -12.], [-10.5, -11.5, -10.], [-10.5, -11.5, -11.])]
loss, margin = dpo_loss(lw, ll, rw, rl); print("DPO: pérdida =", float(loss), "| márgenes implícitos =", margin.numpy().round(3))

In [ ]:
# ---- K.5: SSM lineal: equivalencia recurrencia <-> convolución ----
rng = np.random.default_rng(0); N, T = 8, 64
A = -np.diag(rng.uniform(0.1, 1.0, N)); Bv = rng.normal(size=(N, 1)); Cv = rng.normal(size=(1, N)); dt = 0.1
Abar = np.diag(np.exp(np.diag(A) * dt)); Bbar = np.linalg.solve(A, (Abar - np.eye(N))) @ Bv        # discretización ZOH
x = rng.normal(size=T); h = np.zeros((N, 1)); y_rec = []
for t in range(T): h = Abar @ h + Bbar * x[t]; y_rec.append(float(Cv @ h))
Kc = np.array([float(Cv @ np.linalg.matrix_power(Abar, k) @ Bbar) for k in range(T)])
y_conv = np.array([np.sum(Kc[:t + 1][::-1] * x[:t + 1]) for t in range(T)])
print(f"máx |recurrente − convolución| = {np.max(np.abs(np.array(y_rec) - y_conv)):.2e}   (≈ 0)")

---
## Ejercicios propuestos
**Teóricos**
1. **(T1)** Demuestre que *multi-head attention* y la FFN por token son equivariantes a permutaciones. ¿Qué falla con una máscara de ventana local simétrica? ¿Y con una causal?
2. **(T2)** Si $q_i$ y $k_i$ tienen correlación $\rho$, calcule $\operatorname{Var}(q^\top k)$ y proponga el escalado correcto.
3. **(T3)** Construya un ejemplo con $n=3,\ d_k=1$ de una matriz estocástica que **no** pueda representarse exactamente como $\operatorname{softmax}(S)$.
4. **(T4)** Demuestre que ALiBi es una codificación relativa pura. ¿Por qué extrapola mejor que sumar sinusoides?
5. **(T5)** Pruebe que si $A$ es estocástica por filas y positiva entonces $\operatorname{res}(AX)<\operatorname{res}(X)$ salvo que $X$ ya tenga filas idénticas.
6. **(T6)** Calcule a primer orden en $\|F'\|\ll1$ el jacobiano de un bloque Pre-LN y de uno Post-LN y compare $\|\partial x_L/\partial x_0\|$.
7. **(T7)** Demuestre que la complejidad de IO de FlashAttention es $\Theta(n^2d^2/M)$ para $M\ge d^2$ y compárela con la estándar.
8. **(T8)** Derive $N^\star,D^\star$ si el coste de **inferencia** de $I$ tokens se suma: $C=6ND+2NI$. ¿Qué cambia respecto a Chinchilla? (Ver [31].)
9. **(T9–T10)** Enuncie con precisión qué *no* implica el Teorema 9 y dé un argumento de por qué PARITY es fácil con chain-of-thought.

**Computacionales (TensorFlow)**
10. Implemente **ALiBi** y compare la extrapolación de longitud con RoPE y sinusoidal en el modelo del Bloque D.
11. Añada **KV-cache** al `TinyGPT` y mida el tiempo de generación frente a recomputar.
12. Repita Pre-LN vs Post-LN con $L\in\{4,8,16,32\}$ **con y sin warm-up** y grafique.
13. Entrene un Transformer en **copia con distractores** y verifique la formación de una **induction head**.
14. Implemente **GQA** con $h_{kv}\in\{1,2,4,8\}$ y mida el trade-off calidad/memoria.
15. En el Bloque J añada **ARIMA/ETS** como línea base con *rolling-origin cross-validation*. ¿Sigue ganando el Transformer?

---
## Resumen de la sesión
| # | Resultado | Idea clave | Implicación |
|---|---|---|---|
| **T1** | Atención equivariante a permutaciones | sin PE sólo ve un conjunto | necesidad de codificación posicional |
| **T2** | $\operatorname{Var}(q^\top k)=d_k$ | el escalado evita saturar la softmax | inicialización; QK-Norm |
| **T3** | $\operatorname{rank}(S)\le d_k$ | cuello de botella de rango por cabeza | elección de $d_k$ |
| **T4** | RoPE: el score depende de $n-m$ | rotación ortogonal por pares | posición relativa y extrapolación |
| **T5** | Colapso de rango doblemente exponencial | residuales y MLP lo evitan | por qué hay residuales |
| **T6** | Pre-LN vs Post-LN: gradientes | LN dentro de la rama residual | *warm-up*, profundidad |
| **T7** | Softmax en línea ⇒ atención exacta por bloques | memoria $O(n)$, menos IO | FlashAttention |
| **T8** | $N^\star\propto C^{\beta/(\alpha+\beta)}$ | escalar $N$ y $D$ a la par | diseño compute-óptimo |
| **T9** | Aproximación universal (con FFN y PE) | existencia, no aprendibilidad | alcance de la expresividad |
| **T10** | $\mathsf{TC}^0$ con precisión log. | cota condicional; el CoT la relaja | límites y razonamiento |

> **Para llevarse.** (1) Cada ingrediente de la arquitectura responde a un resultado formal. (2) **Expresividad ≠ aprendibilidad**. (3) En ciencia de datos, **compare siempre con líneas base fuertes**.

---
# Referencias
> **Verificación.** Autores, título, sede, año e identificador arXiv/DOI de los trabajos centrales (Vaswani, Yun, Hahn, Merrill–Sabharwal, Dong, Xiong, Bhojanapalli, Noci, Su, Press, Dao, Milakov–Gimelshein, Rabe–Staats, Kaplan, Hoffmann, Shazeer, Ainslie, Hu, Dettmers, Aghajanyan, Rafailov, Ouyang, Christiano, Gu–Dao, Gu et al., Dosovitskiy, Radford, Fedus, Jiang, Nie, Zeng, Ansari, Gorishniy, Elhage, Olsson, Li et al., Chiang–Cholak) fueron **buscados en línea**. Las demás referencias se citan de memoria: **confirme páginas y números exactos antes de publicar**. Los enunciados de los Teoremas 5, 6, 9 y 10 son **resúmenes**; consulte la fuente para constantes e hipótesis exactas.

## Atención y arquitectura base
- **[1]** Vaswani, A. et al. (2017). *Attention is all you need*. **NeurIPS 30**. arXiv:1706.03762.
- Bahdanau, D., Cho, K. & Bengio, Y. (2015). *Neural machine translation by jointly learning to align and translate*. **ICLR 2015**. arXiv:1409.0473.
- Luong, M.-T., Pham, H. & Manning, C. D. (2015). *Effective approaches to attention-based neural machine translation*. **EMNLP 2015**. arXiv:1508.04025.
- **[2]** Lee, J. et al. (2019). *Set Transformer*. **ICML 2019**. arXiv:1810.00825.
- **[8]** Devlin, J. et al. (2019). *BERT*. **NAACL 2019**. arXiv:1810.04805.
- Raffel, C. et al. (2020). *Exploring the limits of transfer learning with a unified text-to-text transformer* (T5). **JMLR**, 21(140). arXiv:1910.10683.
- Ranzato, M. et al. (2016). *Sequence level training with recurrent neural networks*. **ICLR 2016**. arXiv:1511.06732.

## Propiedades de la autoatención (T1–T3)
- **[3]** Yun, C., Bhojanapalli, S., Rawat, A. S., Reddi, S. J. & Kumar, S. (2020). *Are transformers universal approximators of sequence-to-sequence functions?* **ICLR 2020**. arXiv:1912.10077.
- **[4]** Haviv, A., Ram, O., Press, O., Izsak, P. & Levy, O. (2022). *Transformer language models without positional encodings still learn positional information*. **Findings of EMNLP 2022**. arXiv:2203.16634.
- **[5]** Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.
- **[6]** Dehghani, M. et al. (2023). *Scaling vision transformers to 22 billion parameters*. **ICML 2023**. arXiv:2302.05442.
- **[7]** Bhojanapalli, S., Yun, C., Rawat, A. S., Reddi, S. & Kumar, S. (2020). *Low-rank bottleneck in multi-head attention models*. **ICML 2020**, PMLR 119.

## Codificación posicional (T4)
- **[9]** Shaw, P., Uszkoreit, J. & Vaswani, A. (2018). *Self-attention with relative position representations*. **NAACL 2018**.
- **[10]** Su, J. et al. (2021). *RoFormer: Enhanced transformer with rotary position embedding*. arXiv:2104.09864.
- **[11]** Press, O., Smith, N. A. & Lewis, M. (2022). *Train short, test long: Attention with linear biases enables input length extrapolation*. **ICLR 2022**. arXiv:2108.12409.
- **[12]** Barbero, F. et al. (2025). *Round and round we go! What makes rotary positional encodings useful?* **ICLR 2025**. arXiv:2410.06205.

## Estabilidad y profundidad (T5–T6)
- **[13]** Ainslie, J. et al. (2023). *GQA: Training generalized multi-query transformer models from multi-head checkpoints*. **EMNLP 2023**. arXiv:2305.13245.
- **[14]** Dong, Y., Cordonnier, J.-B. & Loukas, A. (2021). *Attention is not all you need: Pure attention loses rank doubly exponentially with depth*. **ICML 2021**. arXiv:2103.03404.
- **[15]** Noci, L. et al. (2022). *Signal propagation in transformers: Theoretical perspectives and the role of rank collapse*. **NeurIPS 2022**.
- **[16]** Xiong, R. et al. (2020). *On layer normalization in the transformer architecture*. **ICML 2020**. arXiv:2002.04745.
- **[17]** Liu, L., Liu, X., Gao, J., Chen, W. & Han, J. (2020). *Understanding the difficulty of training transformers*. **EMNLP 2020**. arXiv:2004.08249.
- **[18]** Wang, H. et al. (2024). *DeepNet: Scaling transformers to 1,000 layers*. **IEEE TPAMI**. arXiv:2203.00555.

## Atención eficiente e inferencia (T7)
- **[19]** Dao, T., Fu, D. Y., Ermon, S., Rudra, A. & Ré, C. (2022). *FlashAttention: Fast and memory-efficient exact attention with IO-awareness*. **NeurIPS 2022**. arXiv:2205.14135.
- **[20]** Milakov, M. & Gimelshein, N. (2018). *Online normalizer calculation for softmax*. arXiv:1805.02867.
- **[21]** Rabe, M. N. & Staats, C. (2021). *Self-attention does not need $O(n^2)$ memory*. arXiv:2112.05682.
- **[22]** Dao, T. (2023). *FlashAttention-2*. arXiv:2307.08691.
- **[23]** Wang, S. et al. (2020). *Linformer*. arXiv:2006.04768.
- **[24]** Choromanski, K. et al. (2021). *Rethinking attention with Performers*. **ICLR 2021**. arXiv:2009.14794.
- **[25]** Katharopulos, A. et al. (2020). *Transformers are RNNs*. **ICML 2020**. arXiv:2006.16236.
- **[26]** Shazeer, N. (2019). *Fast transformer decoding: One write-head is all you need*. arXiv:1911.02150.

## Escalamiento (T8)
- **[27]** Kaplan, J. et al. (2020). *Scaling laws for neural language models*. arXiv:2001.08361.
- **[28]** Hoffmann, J. et al. (2022). *Training compute-optimal large language models*. arXiv:2203.15556.
- **[29]** Besiroglu, T. et al. (2024). *Chinchilla scaling: A replication attempt*. arXiv:2404.10102.
- **[30]** Muennighoff, N. et al. (2023). *Scaling data-constrained language models*. **NeurIPS 2023**. arXiv:2305.16264.
- **[31]** Sardana, N. et al. (2024). *Beyond Chinchilla-optimal: Accounting for inference in language model scaling laws*. **ICML 2024**. arXiv:2401.00448.

## Expresividad y límites (T9–T10)
- **[32]** Edelman, B. L., Goel, S., Kakade, S. & Zhang, C. (2022). *Inductive biases and variable creation in self-attention mechanisms*. **ICML 2022**. arXiv:2110.10090.
- **[33]** Jiang, H. & Li, Q. (2024). *Approximation rate of the transformer architecture for sequence modeling*. **NeurIPS 2024**. arXiv:2305.18475.
- **[34]** Hahn, M. (2020). *Theoretical limitations of self-attention in neural sequence models*. **TACL**, 8, 156–171.
- **[35]** Merrill, W. & Sabharwal, A. (2023). *The parallelism tradeoff: Limitations of log-precision transformers*. **TACL**, 11, 531–545.
- **[36]** Chiang, D. & Cholak, P. (2022). *Overcoming a theoretical limitation of self-attention*. **ACL 2022**.
- **[37]** Merrill, W. & Sabharwal, A. (2023). *A logic for expressing log-precision transformers*. **NeurIPS 2023**. arXiv:2210.02671.
- **[38]** Merrill, W. & Sabharwal, A. (2024). *The expressive power of transformers with chain of thought*. **ICLR 2024**. arXiv:2310.07923.
- **[39]** Li, Z., Liu, H., Zhou, D. & Ma, T. (2024). *Chain of thought empowers transformers to solve inherently serial problems*. **ICLR 2024**. arXiv:2402.12875.
- **[40]** Strobl, L., Merrill, W., Weiss, G., Chiang, D. & Angluin, D. (2024). *What formal languages can transformers express? A survey*. **TACL**, 12. arXiv:2311.00208.

## Interpretabilidad mecanicista
- **[41]** Elhage, N. et al. (2021). *A mathematical framework for transformer circuits*. Transformer Circuits Thread (Anthropic). https://transformer-circuits.pub/2021/framework/index.html
- **[42]** Olsson, C. et al. (2022). *In-context learning and induction heads*. Transformer Circuits Thread. arXiv:2209.11895.

## Datos estructurados
- **[43]** Zeng, A., Chen, M., Zhang, L. & Xu, Q. (2023). *Are transformers effective for time series forecasting?* **AAAI 2023**. arXiv:2205.13504.
- **[44]** Nie, Y., Nguyen, N. H., Sinthong, P. & Kalagnanam, J. (2023). *A time series is worth 64 words: Long-term forecasting with transformers* (PatchTST). **ICLR 2023**. arXiv:2211.14730.
- **[45]** Ansari, A. F. et al. (2024). *Chronos: Learning the language of time series*. arXiv:2403.07815.
- **[46]** Gorishniy, Y., Rubachev, I., Khrulkov, V. & Babenko, A. (2021). *Revisiting deep learning models for tabular data*. **NeurIPS 2021**. arXiv:2106.11959.
- **[47]** Grinsztajn, L., Oyallon, E. & Varoquaux, G. (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* **NeurIPS 2022 (Datasets and Benchmarks)**. arXiv:2207.08815.
- **[48]** Shwartz-Ziv, R. & Armon, A. (2022). *Tabular data: Deep learning is not all you need*. **Information Fusion**, 81. arXiv:2106.03253.

## Frontera
- **[49]** Fedus, W., Zoph, B. & Shazeer, N. (2022). *Switch transformers*. **JMLR**, 23(120). arXiv:2101.03961.
- **[50]** Jiang, A. Q. et al. (2024). *Mixtral of experts*. arXiv:2401.04088.
- **[51]** Dosovitskiy, A. et al. (2021). *An image is worth 16×16 words*. **ICLR 2021**. arXiv:2010.11929.
- **[52]** Radford, A. et al. (2021). *Learning transferable visual models from natural language supervision* (CLIP). **ICML 2021**. arXiv:2103.00020.
- **[53]** Hu, E. J. et al. (2022). *LoRA: Low-rank adaptation of large language models*. **ICLR 2022**. arXiv:2106.09685.
- **[54]** Aghajanyan, A., Gupta, S. & Zettlemoyer, L. (2021). *Intrinsic dimensionality explains the effectiveness of language model fine-tuning*. **ACL 2021**. arXiv:2012.13255.
- **[55]** Dettmers, T., Pagnoni, A., Holtzman, A. & Zettlemoyer, L. (2023). *QLoRA: Efficient finetuning of quantized LLMs*. **NeurIPS 2023**. arXiv:2305.14314.
- **[56]** Christiano, P. et al. (2017). *Deep reinforcement learning from human preferences*. **NeurIPS 2017**. arXiv:1706.03741.
- **[57]** Ouyang, L. et al. (2022). *Training language models to follow instructions with human feedback*. **NeurIPS 2022**. arXiv:2203.02155.
- **[58]** Rafailov, R. et al. (2023). *Direct preference optimization*. **NeurIPS 2023**. arXiv:2305.18290.
- **[59]** Gu, A., Goel, K. & Ré, C. (2022). *Efficiently modeling long sequences with structured state spaces* (S4). **ICLR 2022**. arXiv:2111.00396.
- **[60]** Gu, A. & Dao, T. (2023). *Mamba: Linear-time sequence modeling with selective state spaces*. arXiv:2312.00752.
- **[61]** Dao, T. & Gu, A. (2024). *Transformers are SSMs*. **ICML 2024**. arXiv:2405.21060.

## Libros y cursos recomendados
- Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing*, 3.ª ed. (borrador). https://web.stanford.edu/~jurafsky/slp3/
- Prince, S. J. D. (2023). *Understanding Deep Learning*. MIT Press. https://udlbook.github.io/udlbook/ (Cap. 12)
- Bishop, C. M. & Bishop, H. (2024). *Deep Learning: Foundations and Concepts*. Springer. (Cap. 12)
- Murphy, K. P. (2023). *Probabilistic Machine Learning: Advanced Topics*. MIT Press.
- Tay, Y. et al. (2022). *Efficient transformers: A survey*. **ACM Computing Surveys**, 55(6). arXiv:2009.06732.
- Stanford CS336 *Language Modeling from Scratch* y CS224N *NLP with Deep Learning* (materiales abiertos).